# Fresh-versus-Rotten Fruit Classification — Training Notebook
**Ramdeobaba University · Deep Learning Lab Capstone (2026–27)**
Team: Priya Keshwani (38) · Pushkar Wankhede (39) · Aastha Pardikar (02)

This notebook trains and compares **two models** on the 6-class task
(Apple / Banana / Orange × Fresh / Rotten):

1. A **baseline custom CNN** (from scratch).
2. **MobileNetV3-Large** via transfer learning (the proposed model).

It produces every metric in the proposal — accuracy, precision, recall, F1,
confusion matrix, per-class performance, **inference latency & FPS** — and
saves the trained MobileNetV3 so the Streamlit app can serve it.

> **Runtime:** go to `Runtime → Change runtime type → T4 GPU` before you start.

## Step 0 — Check the GPU and install dependencies

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU. Set Runtime -> Change runtime type -> T4 GPU, then re-run.")

# torchvision + sklearn + matplotlib come pre-installed on Colab.
# We only need the Kaggle CLI to pull the dataset.
!pip -q install kaggle

## Step 1 — Download the Kaggle dataset

You need a **Kaggle API token**:
1. Go to kaggle.com → your profile → **Settings → API → Create New Token**.
   This downloads a `kaggle.json` file.
2. Run the cell below and **upload that `kaggle.json`** when prompted.

Dataset: `sriramr/fruits-fresh-and-rotten-for-classification`
(~13,600 images, already split into 6 class folders).

In [ ]:
import os, json
from google.colab import files

print("Upload your kaggle.json (from Kaggle -> Settings -> API -> Create New Token)")
uploaded = files.upload()                       # pick kaggle.json

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(uploaded["kaggle.json"])
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("kaggle.json installed.")

In [ ]:
# Download + unzip the dataset (skips if already present)
import os
if not os.path.exists("/content/dataset"):
    !kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification -p /content
    !unzip -q /content/fruits-fresh-and-rotten-for-classification.zip -d /content
print("Contents of /content:")
!ls /content
# The archive extracts to /content/dataset/{train,test}/<6 class folders>
DATA_ROOT = "/content/dataset"
for split in ["train", "test"]:
    p = os.path.join(DATA_ROOT, split)
    if os.path.isdir(p):
        print(split, "->", sorted(os.listdir(p)))

## Step 2 — Build a unified dataset and a 70 / 15 / 15 split

The archive ships a train/test split, but the proposal specifies **70% train /
15% validation / 15% test**. We therefore pool *all* images and make a fresh,
**stratified**, seed-fixed split so every class is proportionally represented.

In [ ]:
import os, glob, random
from collections import Counter

DATA_ROOT = "/content/dataset"
SEED = 42
random.seed(SEED)

# Canonical class order (folder names in the Kaggle dataset)
CLASSES = ["freshapples", "freshbanana", "freshoranges",
           "rottenapples", "rottenbanana", "rottenoranges"]
# Human-readable labels for the app / reports
PRETTY = {
    "freshapples":  "Fresh Apple",  "rottenapples":  "Rotten Apple",
    "freshbanana":  "Fresh Banana", "rottenbanana":  "Rotten Banana",
    "freshoranges": "Fresh Orange", "rottenoranges": "Rotten Orange",
}
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# Gather every image path with its label, from BOTH train/ and test/ folders
all_items = []   # (path, label_idx)
for split in ["train", "test"]:
    for c in CLASSES:
        d = os.path.join(DATA_ROOT, split, c)
        if not os.path.isdir(d):
            continue
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.PNG", "*.JPG", "*.JPEG"):
            for fp in glob.glob(os.path.join(d, ext)):
                all_items.append((fp, class_to_idx[c]))

print("Total images:", len(all_items))
print("Per class:", {CLASSES[i]: n for i, n in sorted(Counter(l for _, l in all_items).items())})

# Stratified 70/15/15 split
from collections import defaultdict
by_class = defaultdict(list)
for fp, lab in all_items:
    by_class[lab].append(fp)

train_items, val_items, test_items = [], [], []
for lab, paths in by_class.items():
    random.shuffle(paths)
    n = len(paths)
    n_tr = int(0.70 * n)
    n_va = int(0.15 * n)
    for fp in paths[:n_tr]:            train_items.append((fp, lab))
    for fp in paths[n_tr:n_tr+n_va]:   val_items.append((fp, lab))
    for fp in paths[n_tr+n_va:]:       test_items.append((fp, lab))

random.shuffle(train_items); random.shuffle(val_items); random.shuffle(test_items)
print(f"Train: {len(train_items)}  Val: {len(val_items)}  Test: {len(test_items)}")

## Step 3 — Datasets, augmentation, and dataloaders

In [ ]:
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Training augmentation: crop, rotation, brightness (as named in the proposal)
train_tf = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
# Deterministic transform for val / test
eval_tf = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

class FruitDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

BATCH = 32
train_ds = FruitDataset(train_items, train_tf)
val_ds   = FruitDataset(val_items,   eval_tf)
test_ds  = FruitDataset(test_items,  eval_tf)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## Step 4 — Shared training & evaluation helpers

In [ ]:
import time, numpy as np
import torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

def train_model(model, train_dl, val_dl, epochs, lr=1e-3, name="model"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_acc, best_state = 0.0, None

    for ep in range(1, epochs + 1):
        model.train(); running = 0.0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward(); optimizer.step()
            running += loss.item() * x.size(0)
        train_loss = running / len(train_dl.dataset)

        # validation
        model.eval(); v_run = 0.0; preds, gts = [], []
        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                out = model(x)
                v_run += criterion(out, y).item() * x.size(0)
                preds += out.argmax(1).cpu().tolist(); gts += y.cpu().tolist()
        val_loss = v_run / len(val_dl.dataset)
        val_acc  = accuracy_score(gts, preds)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"[{name}] epoch {ep:02d}/{epochs}  train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)   # restore best-val weights
    print(f"[{name}] best val_acc = {best_acc:.4f}")
    return model, history

@torch.no_grad()
def evaluate(model, dl, class_names):
    model.eval(); preds, gts = [], []
    for x, y in dl:
        x = x.to(device)
        preds += model(x).argmax(1).cpu().tolist()
        gts += y.tolist()
    acc = accuracy_score(gts, preds)
    p, r, f1, _ = precision_recall_fscore_support(gts, preds, average="macro", zero_division=0)
    report = classification_report(gts, preds, target_names=class_names, zero_division=0)
    cm = confusion_matrix(gts, preds)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1,
            "report": report, "cm": cm, "preds": preds, "gts": gts}

@torch.no_grad()
def measure_latency(model, n=100, warmup=10):
    '''Average single-image inference latency (ms) and FPS.'''
    model.eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    for _ in range(warmup):
        model(dummy)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n):
        model(dummy)
    if device == "cuda":
        torch.cuda.synchronize()
    ms = (time.time() - t0) / n * 1000.0
    return {"latency_ms": ms, "fps": 1000.0 / ms}

## Step 5 — Baseline custom CNN

In [ ]:
import torch.nn as nn

class BaselineCNN(nn.Module):
    '''Simple 4-block CNN trained from scratch — the comparison baseline.'''
    def __init__(self, num_classes=6):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(
            block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, num_classes))
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

cnn = BaselineCNN(num_classes=len(CLASSES))
cnn, cnn_hist = train_model(cnn, train_dl, val_dl, epochs=12, lr=1e-3, name="BaselineCNN")

## Step 6 — MobileNetV3-Large (transfer learning)

In [ ]:
import torchvision
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

def build_mobilenetv3(num_classes=6, pretrained=True):
    weights = MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None
    model = mobilenet_v3_large(weights=weights)
    in_f = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_f, num_classes)   # 6-class head
    return model

mnet = build_mobilenetv3(num_classes=len(CLASSES), pretrained=True)
# Fine-tune the whole network (lower LR because the backbone is pretrained)
mnet, mnet_hist = train_model(mnet, train_dl, val_dl, epochs=8, lr=3e-4, name="MobileNetV3")

## Step 7 — Evaluation on the held-out test set (both models)

In [ ]:
import matplotlib.pyplot as plt

pretty_names = [PRETTY[c] for c in CLASSES]

cnn_res  = evaluate(cnn,  test_dl, pretty_names)
mnet_res = evaluate(mnet, test_dl, pretty_names)
cnn_lat  = measure_latency(cnn)
mnet_lat = measure_latency(mnet)

print("===== Baseline CNN =====")
print(cnn_res["report"])
print(f"Latency: {cnn_lat['latency_ms']:.2f} ms   FPS: {cnn_lat['fps']:.1f}")
print("\n===== MobileNetV3 =====")
print(mnet_res["report"])
print(f"Latency: {mnet_lat['latency_ms']:.2f} ms   FPS: {mnet_lat['fps']:.1f}")

# Comparison table
print("\n================ COMPARISON ================")
print(f"{'Metric':<16}{'BaselineCNN':>14}{'MobileNetV3':>14}")
for k in ["accuracy", "precision", "recall", "f1"]:
    print(f"{k:<16}{cnn_res[k]:>14.4f}{mnet_res[k]:>14.4f}")
print(f"{'latency_ms':<16}{cnn_lat['latency_ms']:>14.2f}{mnet_lat['latency_ms']:>14.2f}")
print(f"{'fps':<16}{cnn_lat['fps']:>14.1f}{mnet_lat['fps']:>14.1f}")

In [ ]:
# Confusion matrices
def plot_cm(cm, names, title, ax):
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_cm(cnn_res["cm"],  pretty_names, "Baseline CNN", axes[0])
plot_cm(mnet_res["cm"], pretty_names, "MobileNetV3",  axes[1])
plt.tight_layout(); plt.savefig("/content/confusion_matrices.png", dpi=120); plt.show()

## Step 8 — Save artifacts and download

We save the **MobileNetV3** weights (the model the app will serve), the label
map, the metrics, and the confusion-matrix figure. Everything is zipped and
downloaded to your computer — drop `mobilenetv3_fresh_rotten.pt` and
`labels.json` into the `app/models/` folder of the project.

In [ ]:
import json, torch

torch.save(mnet.state_dict(), "/content/mobilenetv3_fresh_rotten.pt")
# (optional) also keep the baseline for the report
torch.save(cnn.state_dict(), "/content/baseline_cnn.pt")

with open("/content/labels.json", "w") as f:
    json.dump({"classes": CLASSES,
               "pretty": [PRETTY[c] for c in CLASSES],
               "arch": "mobilenet_v3_large",
               "img_size": IMG_SIZE, "mean": MEAN, "std": STD}, f, indent=2)

metrics = {
    "baseline_cnn": {k: cnn_res[k] for k in ["accuracy","precision","recall","f1"]} | cnn_lat,
    "mobilenetv3":  {k: mnet_res[k] for k in ["accuracy","precision","recall","f1"]} | mnet_lat,
    "split": {"train": len(train_items), "val": len(val_items), "test": len(test_items)},
}
with open("/content/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

!cd /content && zip -q artifacts.zip mobilenetv3_fresh_rotten.pt baseline_cnn.pt labels.json metrics.json confusion_matrices.png
from google.colab import files
files.download("/content/artifacts.zip")
print("Done. Unzip artifacts.zip and copy mobilenetv3_fresh_rotten.pt + labels.json into app/models/")